# Partie 3 — GOLD :
## Objectif
Construire des **tables analytiques Gold** à partir des tables **Silver** déjà nettoyées.

Nous allons produire quatre tables :

| Table Gold | Indicateur calculé |
|---|---|
| `medical.gold.consultations_par_categorie` | Nombre de consultations et de patients distincts par catégorie |
| `medical.gold.activite_mensuelle` | Nombre de consultations et de patients distincts par mois |
| `medical.gold.analyses_par_type` | Nombre d'observations médicales par catégorie, code et type |
| `medical.gold.patients_par_genre` | Répartition des patients par genre |





In [1]:
# 1. Démarrer la session Spark de la partie Gold
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("MedicalLakehouse-Partie3-Gold")
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config(
        "spark.sql.catalog.medical",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config("spark.sql.catalog.medical.type", "rest")
    .config("spark.sql.catalog.medical.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.medical.warehouse", "s3://medical-warehouse/")
    .config("spark.sql.catalog.medical.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.medical.s3.endpoint", "http://rustfs:9000")
    .config("spark.sql.catalog.medical.s3.path-style-access", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark :", spark.version)
print("Catalogue REST :", spark.conf.get("spark.sql.catalog.medical.uri"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/19 10:50:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/19 10:50:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/19 10:50:52 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/10/19 10:50:52 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/10/19 10:50:52 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


Spark : 3.5.5
Catalogue REST : http://iceberg-rest:8181


## 2 — Vérifier les tables Silver

In [2]:
# 2. Vérifier les trois tables Silver
tables_silver = {
    "patients": "medical.silver.patients",
    "consultations": "medical.silver.consultations",
    "observations": "medical.silver.observations",
}

for nom, table in tables_silver.items():
    assert spark.catalog.tableExists(table), (
        f"Table manquante : {table}. Exécutez d'abord le notebook Silver."
    )
    print(f"{nom:15} : {spark.table(table).count()} lignes")

# Charger les tables Silver dans ce notebook
df_patients = spark.table(tables_silver["patients"])
df_consultations = spark.table(tables_silver["consultations"])
df_observations = spark.table(tables_silver["observations"])

# Vérifier les colonnes nécessaires aux indicateurs
colonnes_attendues = {
    "patients": {"patient_id", "gender", "birthdate", "city"},
    "consultations": {"consultation_id", "patient_id", "date_debut", "categorie"},
    "observations": {
        "observation_id", "patient_id", "categorie", "code",
        "description", "type_observation"
    },
}

for nom, df in [
    ("patients", df_patients),
    ("consultations", df_consultations),
    ("observations", df_observations),
]:
    manquantes = colonnes_attendues[nom] - set(df.columns)
    assert not manquantes, f"Colonnes Silver manquantes pour {nom} : {manquantes}"

assert df_patients.count() > 0, "Silver patients est vide."
assert df_consultations.count() > 0, "Silver consultations est vide."
assert df_observations.count() > 0, "Silver observations est vide."

print("Toutes les sources Silver sont disponibles.")

patients        : 108 lignes
consultations   : 5571 lignes
observations    : 68630 lignes
Toutes les sources Silver sont disponibles.


## 3 — Créer le namespace Gold

Le catalogue `medical` regroupe les namespaces `bronze`, `silver`, `quarantine` et désormais `gold`.

Toutes les tables Gold seront enregistrées en **Iceberg**, dans le bucket RustFS `medical-warehouse`.

In [3]:
# 3. Créer l'espace Gold
spark.sql("CREATE NAMESPACE IF NOT EXISTS medical.gold")

# Afficher les namespaces existants
spark.sql("SHOW NAMESPACES IN medical").show(truncate=False)

+----------+
|namespace |
+----------+
|bronze    |
|quarantine|
|sandbox   |
|silver    |
|gold      |
+----------+



## 4 — Fonction simple pour enregistrer une table Gold

Cette fonction crée la table si elle n'existe pas.

**Protection contre les doublons :** si le notebook est relancé, la table Gold existante n'est pas écrasée. Si les données Silver ont changé depuis sa création, il faudra choisir explicitement une procédure de mise à jour ou de reconstruction ; ce notebook **ne remplace pas automatiquement** les anciens résultats.

In [4]:
# 4. Créer une table Iceberg uniquement si elle n'existe pas
def enregistrer_gold(df, nom_table):
    if spark.catalog.tableExists(nom_table):
        print("Table existante, aucune écriture :", nom_table)
        print("Remarque : vérifier sa fraîcheur si Silver a été mis à jour.")
    else:
        df.writeTo(nom_table).using("iceberg").create()
        print("Table Iceberg créée :", nom_table)

    print("Nombre de lignes Gold :", spark.table(nom_table).count())

## 5 — Indicateur 1 : consultations par catégorie

Nous calculons :

- le **nombre de consultations** pour chaque catégorie Synthea ;
- le **nombre de patients distincts** ayant eu une consultation dans cette catégorie.

Une même personne peut être comptée dans plusieurs catégories. La somme des *patients distincts par catégorie* n'est donc pas nécessairement égale au nombre total de patients.

Les valeurs de catégorie absentes sont regroupées sous le libellé `Non renseignée`.

In [5]:
# 5. Calculer les consultations par catégorie
df_gold_categories = spark.sql("""
    WITH consultations AS (
        SELECT
            consultation_id,
            patient_id,
            COALESCE(NULLIF(TRIM(categorie), ''), 'Non renseignée') AS categorie
        FROM medical.silver.consultations
    )
    SELECT
        categorie,
        COUNT(*) AS nombre_consultations,
        COUNT(DISTINCT patient_id) AS nombre_patients
    FROM consultations
    GROUP BY categorie
""")

df_gold_categories.orderBy(
    F.desc("nombre_consultations")
).show(20, truncate=False)

[Stage 18:>                                                         (0 + 1) / 1]

+----------+--------------------+---------------+
|categorie |nombre_consultations|nombre_patients|
+----------+--------------------+---------------+
|ambulatory|2939                |106            |
|wellness  |1262                |108            |
|outpatient|759                 |101            |
|emergency |270                 |83             |
|urgentcare|195                 |35             |
|home      |63                  |3              |
|inpatient |58                  |33             |
|snf       |9                   |7              |
|virtual   |8                   |6              |
|hospice   |8                   |8              |
+----------+--------------------+---------------+



In [6]:
# 6. Enregistrer la première table Gold
enregistrer_gold(
    df_gold_categories,
    "medical.gold.consultations_par_categorie"
)

spark.sql("""
    SELECT *
    FROM medical.gold.consultations_par_categorie
    ORDER BY nombre_consultations DESC
""").show(20, truncate=False)

Table Iceberg créée : medical.gold.consultations_par_categorie
Nombre de lignes Gold : 10
+----------+--------------------+---------------+
|categorie |nombre_consultations|nombre_patients|
+----------+--------------------+---------------+
|ambulatory|2939                |106            |
|wellness  |1262                |108            |
|outpatient|759                 |101            |
|emergency |270                 |83             |
|urgentcare|195                 |35             |
|home      |63                  |3              |
|inpatient |58                  |33             |
|snf       |9                   |7              |
|virtual   |8                   |6              |
|hospice   |8                   |8              |
+----------+--------------------+---------------+



## 6 — Indicateur 2 : évolution mensuelle de l'activité

Nous regroupons les consultations par **mois de début**, au format `AAAA-MM`.

Le `date_debut` provient de Silver : il a déjà été converti en `TIMESTAMP`. La session Spark utilise le fuseau `UTC`.

Nous calculons le nombre de consultations et le nombre de patients distincts par mois.

In [7]:
# 7. Calculer l'activité mensuelle
df_gold_mensuel = spark.sql("""
    SELECT
        DATE_FORMAT(date_debut, 'yyyy-MM') AS mois,
        COUNT(*) AS nombre_consultations,
        COUNT(DISTINCT patient_id) AS nombre_patients
    FROM medical.silver.consultations
    WHERE date_debut IS NOT NULL
    GROUP BY DATE_FORMAT(date_debut, 'yyyy-MM')
""")

df_gold_mensuel.orderBy("mois").show(50, truncate=False)

+-------+--------------------+---------------+
|mois   |nombre_consultations|nombre_patients|
+-------+--------------------+---------------+
|1942-11|1                   |1              |
|1947-01|1                   |1              |
|1948-12|1                   |1              |
|1950-01|1                   |1              |
|1953-01|1                   |1              |
|1955-12|2                   |1              |
|1956-11|1                   |1              |
|1960-03|1                   |1              |
|1960-04|3                   |2              |
|1960-07|1                   |1              |
|1960-09|1                   |1              |
|1960-12|1                   |1              |
|1961-03|1                   |1              |
|1961-06|1                   |1              |
|1961-07|1                   |1              |
|1961-08|1                   |1              |
|1962-12|1                   |1              |
|1963-12|1                   |1              |
|1965-09|2   

In [8]:
# 8. Enregistrer la table Gold mensuelle
enregistrer_gold(
    df_gold_mensuel,
    "medical.gold.activite_mensuelle"
)

spark.sql("""
    SELECT *
    FROM medical.gold.activite_mensuelle
    ORDER BY mois
""").show(50, truncate=False)

Table Iceberg créée : medical.gold.activite_mensuelle
Nombre de lignes Gold : 459
+-------+--------------------+---------------+
|mois   |nombre_consultations|nombre_patients|
+-------+--------------------+---------------+
|1942-11|1                   |1              |
|1947-01|1                   |1              |
|1948-12|1                   |1              |
|1950-01|1                   |1              |
|1953-01|1                   |1              |
|1955-12|2                   |1              |
|1956-11|1                   |1              |
|1960-03|1                   |1              |
|1960-04|3                   |2              |
|1960-07|1                   |1              |
|1960-09|1                   |1              |
|1960-12|1                   |1              |
|1961-03|1                   |1              |
|1961-06|1                   |1              |
|1961-07|1                   |1              |
|1961-08|1                   |1              |
|1962-12|1               

## 7 — Indicateur 3 : observations et analyses par type

Une **observation** Synthea peut être un examen de laboratoire, une mesure physiologique ou une autre observation clinique.

Nous la regroupons par :

- sa catégorie ;
- son code ;
- sa description ;
- son type de valeur (`type_observation`).

Nous calculons le nombre d'observations et le nombre de patients concernés.

**Précaution médicale :** nous ne calculons pas de moyenne de résultats cliniques : mélanger différentes unités ou natures d'observation pourrait produire des indicateurs trompeurs.

In [9]:
# 9. Calculer le nombre d'observations par type
df_gold_analyses = spark.sql("""
    WITH observations AS (
        SELECT
            patient_id,
            COALESCE(NULLIF(TRIM(categorie), ''), 'Non renseignée') AS categorie,
            code,
            COALESCE(NULLIF(TRIM(description), ''), 'Sans description') AS description,
            COALESCE(NULLIF(TRIM(type_observation), ''), 'Non renseigné') AS type_observation
        FROM medical.silver.observations
    )
    SELECT
        categorie,
        code,
        description,
        type_observation,
        COUNT(*) AS nombre_observations,
        COUNT(DISTINCT patient_id) AS nombre_patients
    FROM observations
    GROUP BY categorie, code, description, type_observation
""")

df_gold_analyses.orderBy(
    F.desc("nombre_observations")
).show(15, truncate=False)

+--------------+-------+---------------------------------------------------------------------------------------------------------------------------------------+----------------+-------------------+---------------+
|categorie     |code   |description                                                                                                                            |type_observation|nombre_observations|nombre_patients|
+--------------+-------+---------------------------------------------------------------------------------------------------------------------------------------+----------------+-------------------+---------------+
|vital-signs   |72514-3|Pain severity - 0-10 verbal numeric rating [Score] - Reported                                                                          |numeric         |1808               |108            |
|vital-signs   |8480-6 |Systolic Blood Pressure                                                                                                 

In [10]:
# 10. Enregistrer les analyses dans Gold
enregistrer_gold(
    df_gold_analyses,
    "medical.gold.analyses_par_type"
)

spark.sql("""
    SELECT categorie, code, description,
           type_observation, nombre_observations, nombre_patients
    FROM medical.gold.analyses_par_type
    ORDER BY nombre_observations DESC
    LIMIT 15
""").show(truncate=False)

Table Iceberg créée : medical.gold.analyses_par_type
Nombre de lignes Gold : 252
+--------------+-------+---------------------------------------------------------------------------------------------------------------------------------------+----------------+-------------------+---------------+
|categorie     |code   |description                                                                                                                            |type_observation|nombre_observations|nombre_patients|
+--------------+-------+---------------------------------------------------------------------------------------------------------------------------------------+----------------+-------------------+---------------+
|vital-signs   |72514-3|Pain severity - 0-10 verbal numeric rating [Score] - Reported                                                                          |numeric         |1808               |108            |
|vital-signs   |8480-6 |Systolic Blood Pressure                

## 8 — Indicateur 4 : répartition des patients par genre

Cet indicateur est un exemple simple de **dimension démographique** pour un tableau de bord.

Il compte les patients par valeur de `gender` issue de Silver, sans déduire d'autre information personnelle ou médicale.

In [11]:
# 11. Compter les patients par genre
df_gold_genre = spark.sql("""
    WITH patients AS (
        SELECT
            COALESCE(NULLIF(TRIM(gender), ''), 'Non renseigné') AS genre
        FROM medical.silver.patients
    )
    SELECT
        genre,
        COUNT(*) AS nombre_patients
    FROM patients
    GROUP BY genre
""")

df_gold_genre.orderBy(F.desc("nombre_patients")).show(truncate=False)

+-----+---------------+
|genre|nombre_patients|
+-----+---------------+
|M    |57             |
|F    |51             |
+-----+---------------+



In [12]:
# 12. Enregistrer la table démographique
enregistrer_gold(
    df_gold_genre,
    "medical.gold.patients_par_genre"
)

spark.sql("""
    SELECT *
    FROM medical.gold.patients_par_genre
    ORDER BY nombre_patients DESC
""").show(truncate=False)

Table Iceberg créée : medical.gold.patients_par_genre
Nombre de lignes Gold : 2
+-----+---------------+
|genre|nombre_patients|
+-----+---------------+
|M    |57             |
|F    |51             |
+-----+---------------+



## 9 — Vérifier toutes les tables Gold

Nous devons retrouver les quatre tables dans `medical.gold`.

In [13]:
# 13. Afficher les tables Gold
spark.sql("SHOW TABLES IN medical.gold").show(truncate=False)

# Vérifier les quatre tables attendues
tables_gold = [
    "medical.gold.consultations_par_categorie",
    "medical.gold.activite_mensuelle",
    "medical.gold.analyses_par_type",
    "medical.gold.patients_par_genre",
]

for nom_table in tables_gold:
    assert spark.catalog.tableExists(nom_table), f"Table Gold absente : {nom_table}"
    print(nom_table, ":", spark.table(nom_table).count(), "lignes")

+---------+---------------------------+-----------+
|namespace|tableName                  |isTemporary|
+---------+---------------------------+-----------+
|gold     |activite_mensuelle         |false      |
|gold     |analyses_par_type          |false      |
|gold     |consultations_par_categorie|false      |
|gold     |patients_par_genre         |false      |
+---------+---------------------------+-----------+

medical.gold.consultations_par_categorie : 10 lignes
medical.gold.activite_mensuelle : 459 lignes
medical.gold.analyses_par_type : 252 lignes
medical.gold.patients_par_genre : 2 lignes


## 10 — Contrôles des résultats

Vérifier que les nombres agrégés correspondent aux données Silver.

Exemple : **la somme des consultations par catégorie** doit être égale au nombre total de consultations Silver, car chaque consultation appartient à un seul groupe.

Si le contrôle échoue alors que la table Gold existe déjà, elle peut contenir des résultats plus anciens.

In [14]:
# 14. Vérifier que les agrégats représentent bien les lignes Silver
nb_consultations_silver = df_consultations.count()
nb_observations_silver = df_observations.count()
nb_patients_silver = df_patients.count()

total_categories = spark.sql("""
    SELECT SUM(nombre_consultations) AS total
    FROM medical.gold.consultations_par_categorie
""").first()["total"]

total_mensuel = spark.sql("""
    SELECT SUM(nombre_consultations) AS total
    FROM medical.gold.activite_mensuelle
""").first()["total"]

total_analyses = spark.sql("""
    SELECT SUM(nombre_observations) AS total
    FROM medical.gold.analyses_par_type
""").first()["total"]

total_genre = spark.sql("""
    SELECT SUM(nombre_patients) AS total
    FROM medical.gold.patients_par_genre
""").first()["total"]

print("Consultations Silver / Gold catégorie :", nb_consultations_silver, "/", total_categories)
print("Consultations Silver / Gold mensuel   :", nb_consultations_silver, "/", total_mensuel)
print("Observations Silver / Gold type      :", nb_observations_silver, "/", total_analyses)
print("Patients Silver / Gold genre         :", nb_patients_silver, "/", total_genre)

assert total_categories == nb_consultations_silver, "Écart : consultations par catégorie"
assert total_mensuel == nb_consultations_silver, "Écart : consultations mensuelles"
assert total_analyses == nb_observations_silver, "Écart : observations par type"
assert total_genre == nb_patients_silver, "Écart : patients par genre"

print("Tous les contrôles de volume Gold sont OK.")

Consultations Silver / Gold catégorie : 5571 / 5571
Consultations Silver / Gold mensuel   : 5571 / 5571
Observations Silver / Gold type      : 68630 / 68630
Patients Silver / Gold genre         : 108 / 108
Tous les contrôles de volume Gold sont OK.


## 11 — Tableau de bord : quelques chiffres clés

Cette cellule affiche des indicateurs globaux directement depuis Silver :

- nombre de patients ;
- nombre de consultations ;
- nombre d'observations.

Les **tables Gold** construites plus haut détaillent ces résultats par catégorie, mois et type.

In [15]:
# 15. Afficher trois chiffres clés
print("========== TABLEAU DE BORD MÉDICAL ==========")
print("Patients valides      :", nb_patients_silver)
print("Consultations valides :", nb_consultations_silver)
print("Observations valides  :", nb_observations_silver)

print("\nTop 5 des catégories de consultations :")
spark.sql("""
    SELECT categorie, nombre_consultations
    FROM medical.gold.consultations_par_categorie
    ORDER BY nombre_consultations DESC
    LIMIT 5
""").show(truncate=False)

========== TABLEAU DE BORD MÉDICAL ==========
Patients valides      : 108
Consultations valides : 5571
Observations valides  : 68630

Top 5 des catégories de consultations :
+----------+--------------------+
|categorie |nombre_consultations|
+----------+--------------------+
|ambulatory|2939                |
|wellness  |1262                |
|outpatient|759                 |
|emergency |270                 |
|urgentcare|195                 |
+----------+--------------------+



## 12 — Rappel : que deviennent les données de Quarantine ?

Les données **Quarantine** ont été écartées lors du contrôle de qualité dans Silver.

Elles restent consultables pour analyse ou correction, **mais ne sont pas prises en compte dans les agrégats Gold**. Une correction nécessite un retraitement explicite.

In [16]:
# 16. Voir les quantités de données rejetées (sans les intégrer à Gold)
for nom in ["patients", "consultations", "observations"]:
    table = f"medical.quarantine.{nom}"
    if spark.catalog.tableExists(table):
        print(nom, "— rejetés :", spark.table(table).count())
    else:
        print(nom, "— table Quarantine absente")

patients — rejetés : 0
consultations — rejetés : 0
observations — rejetés : 18


## 13 — Vérifier Iceberg et RustFS

Les tables Gold sont de **vraies tables Iceberg**.

- Le catalogue REST garde les références nécessaires.
- RustFS conserve les fichiers Parquet et les métadonnées.
- Les snapshots enregistrent l'historique des modifications de la table.

La console RustFS est accessible depuis Windows à : http://localhost:9001

**Ne modifiez pas manuellement les fichiers Iceberg dans le bucket.**

In [17]:
# 17. Afficher les fichiers de la première table Gold
spark.sql("""
    SELECT file_path, record_count, file_size_in_bytes
    FROM medical.gold.consultations_par_categorie.files
""").show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------+------------+------------------+
|file_path                                                                                                                 |record_count|file_size_in_bytes|
+--------------------------------------------------------------------------------------------------------------------------+------------+------------------+
|s3://medical-warehouse/gold/consultations_par_categorie/data/00000-17-8226fa87-dfb9-4dbf-8781-3447f0682233-0-00001.parquet|10          |1186              |
+--------------------------------------------------------------------------------------------------------------------------+------------+------------------+



In [18]:
# 18. Afficher les snapshots de la première table Gold
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM medical.gold.consultations_par_categorie.snapshots
    ORDER BY committed_at DESC
""").show(truncate=False)

+-------------------+----------------------+---------+
|snapshot_id        |committed_at          |operation|
+-------------------+----------------------+---------+
|1127280944782587581|2026-10-19 10:51:27.45|append   |
+-------------------+----------------------+---------+

